# Analysebeispiele

Dieses Notebook zeigt anhand ausgewählter Beispiele, wie die durch die Pipeline aufbereiteten und transformierten Daten für weiterführende Analysen
verwendet werden können.

Der Schwerpunkt liegt dabei nicht auf einer vollständigen Analyse des Datensatzes, sondern darauf, die praktische Nutzbarkeit der erzeugten Tabellen
und der während der Transformation vorbereiteten Merkmale zu demonstrieren.

## Vorbereitung

Für die Analysebeispiele werden die während der Transformation erzeugten Parquet-Dateien geladen.

In [ ]:
from src.paths import TRANSFORMED_DATA_DIR, IMAGES_DIR
from matplotlib.ticker import EngFormatter
import matplotlib.pyplot as plt
import pandas as pd

date_dataset = pd.read_parquet(TRANSFORMED_DATA_DIR / "date.parquet")

invoice_dataset = pd.read_parquet(TRANSFORMED_DATA_DIR / "invoice.parquet")

invoice_position_dataset = pd.read_parquet(TRANSFORMED_DATA_DIR / "invoice_position.parquet")

Eine globale Matplotlib-Konfiguration wire definiert, um für die folgenden Visualisierungen eine einheitliche Darstellung zu verwenden.

In [ ]:
plt.style.use("default")

plt.rcParams.update({
    "figure.figsize": (16, 8),
    "font.size": 12,
    "axes.titlesize": 18,
    "axes.titleweight": "bold",
    "axes.titlepad": 20,
    "axes.labelsize": 14,
    "axes.labelweight": "bold",
    "axes.labelpad": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 100,
})

## Monatliche Entwicklung zentraler Kennzahlen

Als erstes Beispiel wird die zeitliche Entwicklung von Umsatz, Absatz, Anzahl der Bestellungen und durchschnittlichem Bestellwert betrachtet.

Dazu wird die Rechnungstabelle mit der Datumstabelle verknüpft. Über das Rechnungsdatum können dadurch die vorbereiteten Kalendermerkmale der
Datumstabelle, insbesondere **YearMonth**, direkt für die monatliche Gruppierung verwendet werden.

Da der letzte im Datensatz enthaltene Monat nicht vollständig vorliegt, wird dieser vor der Aggregation ausgeschlossen.

In [ ]:
invoice_date_merge = pd.merge(
    invoice_dataset,
    date_dataset,
    left_on="Date",
    right_index=True,
    how="inner"
)

incomplete_last_month = invoice_date_merge["YearMonth"].max()

invoice_date_merge = invoice_date_merge[invoice_date_merge["YearMonth"] < incomplete_last_month]

invoice_date_merge_group = invoice_date_merge.groupby("YearMonth")

revenue_by_yearmonth = invoice_date_merge_group["TotalRevenue"].sum()

width, height = plt.rcParams["figure.figsize"]

fig, axis = plt.subplots(4, 1, sharex=True, figsize=(width, height * 4))

revenue_by_yearmonth.plot(
    ax=axis[0],
    title="Monthly Revenue",
    xlabel="Month",
    ylabel="Revenue",
    color="darkblue",
    marker="o",
)

quantity_by_yearmonth = invoice_date_merge_group["TotalQuantity"].sum()

quantity_by_yearmonth.plot(
    ax=axis[1],
    title="Monthly Quantity",
    xlabel="Month",
    ylabel="Quantity",
    color="darkgreen",
    marker="o",
)

monthly_order_count = invoice_date_merge_group.size()

monthly_order_count.plot(
    ax=axis[2],
    title="Monthly Order Count",
    xlabel="Month",
    ylabel="Order Count",
    color="darkred",
    marker="o",
)

average_order_value = revenue_by_yearmonth / monthly_order_count

average_order_value.plot(
    ax=axis[3],
    title="Monthly Average Order Value",
    xlabel="Month",
    ylabel="Average Order Value",
    color="black",
    marker="o",
)

for ax in axis:
    ax.yaxis.set_major_formatter(EngFormatter())
    ax.set_xlim(right=revenue_by_yearmonth.index.max() + pd.Timedelta(days=30))

fig.subplots_adjust(hspace=0.2)

fig.savefig(IMAGES_DIR / "monthly_business_metrics.png", bbox_inches="tight")

![Monthly Business Metrics](../images/monthly_business_metrics.png)

## Verteilung der Bestellkennzahlen

Als zweites Beispiel wird die Verteilung der auf Rechnungsebene aggregierten Kennzahlen **TotalRevenue**, **TotalQuantity** und **PositionCount** betrachtet.

Die Kennzahlen beschreiben den Gesamtumsatz, die Gesamtmenge und die Anzahl der Positionen einer Bestellung und wurden bereits während der Transformation
aus den einzelnen Rechnungspositionen abgeleitet.

Da wenige sehr hohe Werte die Darstellung der Verteilungen stark strecken und die übrigen Werte dadurch im Histogramm kaum noch erkennbar wären, werden für die Visualisierung jeweils nur Bestellungen bis zum 99. Perzentil dargestellt. Die Medianwerte werden zusätzlich als gestrichelte Linien eingezeichnet.

In [ ]:
fig, axis = plt.subplots(3, 1, sharex=False, figsize=(width, height * 3))

percentile = 99

for ax in axis:
    ax.text(
        0.5, 1.0,
        f"Orders up to the {percentile}th percentile",
        transform=ax.transAxes,
        ha="center"
    )

upper_limit = invoice_dataset[["TotalRevenue", "TotalQuantity", "PositionCount"]].quantile(percentile / 100)

revenue_filtered = invoice_dataset[invoice_dataset["TotalRevenue"] <= upper_limit["TotalRevenue"]]["TotalRevenue"]

median_revenue = invoice_dataset["TotalRevenue"].median()

axis[0].axvline(
    median_revenue,
    linestyle="--",
    linewidth=1.5,
    color="darkorange"
)

revenue_filtered.plot.hist(
    ax=axis[0],
    bins=50,
    title="Distribution of Order Revenue",
    ylabel="Frequency",
    xlabel="Order Revenue",
    color="steelblue",
    alpha=0.7,
)

quantity_filtered = invoice_dataset[invoice_dataset["TotalQuantity"] <= upper_limit["TotalQuantity"]]["TotalQuantity"]

median_quantity = invoice_dataset["TotalQuantity"].median()

axis[1].axvline(
    median_quantity,
    linestyle="--",
    linewidth=1.5,
    color="steelblue"
)

quantity_filtered.plot.hist(
    ax=axis[1],
    bins=50,
    title="Distribution of Order Quantity",
    xlabel="Order Quantity",
    ylabel="Frequency",
    color="darkorange",
    alpha=0.7,
)

position_count_filtered = invoice_dataset[invoice_dataset["PositionCount"] <= upper_limit["PositionCount"]]["PositionCount"]

median_position_count = invoice_dataset["PositionCount"].median()

axis[2].axvline(
    median_position_count,
    linestyle="--",
    linewidth=1.5,
    color="darkgreen"
)

position_count_filtered.plot.hist(
    ax=axis[2],
    bins=50,
    title="Distribution of Order Position Count",
    xlabel="Order Position Count",
    ylabel="Frequency",
    color="darkred",
    alpha=0.7,
)

fig.subplots_adjust(hspace=0.3)

fig.savefig(IMAGES_DIR / "order_distributions.png", bbox_inches="tight")

![Order Distributions](../images/order_distributions.png)

## Umsatz- und absatzstärkste Produkte

Um zu untersuchen, welche Produkte besonders stark zum Umsatz und zum Absatz beitragen, werden **Revenue** und **Quantity** je Produkt summiert und die jeweils
zehn höchsten Werte dargestellt.

Die beiden Rankings unterscheiden sich deutlich. **REGENCY CAKESTAND 3 TIER** erzielt den höchsten Umsatz, gehört aber nicht zu den zehn meistverkauften
Produkten. Umgekehrt weist **WORLD WAR 2 GLIDERS ASSTD DESIGNS** die höchste verkaufte Menge auf, gehört jedoch nicht zu den zehn umsatzstärksten Produkten.

Daneben gibt es Produkte wie **JUMBO BAG RED RETROSPOT** und **CREAM HANGING HEART T-LIGHT HOLDER**, die in beiden Rankings weit oben liegen.

Eine hohe verkaufte Menge führt somit nicht automatisch zu einem entsprechend hohen Gesamtumsatz.

In [ ]:
product_totals = invoice_position_dataset.groupby(["StockCode", "Description"])[["Revenue", "Quantity"]].sum()
top10_products_by_revenue = product_totals["Revenue"].nlargest(10).sort_values(ascending=True)
top10_products_by_quantity = product_totals["Quantity"].nlargest(10).sort_values(ascending=True)

fig, axis = plt.subplots(2, 1, sharex=False, figsize=(width, height * 2))

for ax in axis:
    ax.xaxis.set_major_formatter(EngFormatter())

top10_products_by_revenue.plot.barh(
    ax=axis[0],
    title="Top 10 Products by Revenue",
    xlabel="Total Revenue",
    ylabel="Products",
    color="steelblue",
    alpha=0.7
)

top10_products_by_quantity.plot.barh(
    ax=axis[1],
    title="Top 10 Products by Quantity",
    xlabel="Total Quantity",
    ylabel="Products",
    color="darkorange",
    alpha=0.7
)

fig.subplots_adjust(hspace=0.3)

fig.savefig(IMAGES_DIR / "top10_products.png", bbox_inches="tight")

![Top 10 Products](../images/top10_products.png)

## Umsatz und Absatz nach Ländern

Um zu sehen, aus welchen Ländern der größte Teil des Geschäfts stammt, werden **Revenue** und **Quantity** nach Ländern summiert und die jeweils zehn größten Werte dargestellt.

Das **United Kingdom** liegt bei Umsatz und verkaufter Menge mit großem Abstand an erster Stelle. Die übrigen Länder machen im Vergleich nur einen kleinen Teil des Gesamtgeschäfts aus.

Unter den übrigen Ländern unterscheiden sich die beiden Rangfolgen. **EIRE** erzielt den zweithöchsten Umsatz, während die **Netherlands** bei der verkauften Menge auf dem zweiten Platz liegen.
Auch bei den weiteren Ländern zeigt sich, dass eine höhere verkaufte Menge nicht automatisch mit einem entsprechend höheren Umsatz verbunden ist.

In [ ]:
invoice_positions_joined = invoice_position_dataset.join(invoice_dataset, how="inner")

country_totals = invoice_positions_joined.groupby("Country")[["Revenue", "Quantity"]].sum()

top10_countries_by_revenue = country_totals["Revenue"].nlargest(10).sort_values(ascending=True)
top10_countries_by_quantity = country_totals["Quantity"].nlargest(10).sort_values(ascending=True)

fig, axis = plt.subplots(2, 1, sharex=False, figsize=(width, height * 2))

for ax in axis:
    ax.xaxis.set_major_formatter(EngFormatter())

top10_countries_by_revenue.plot.barh(
    ax=axis[0],
    title="Top 10 Countries by Revenue",
    xlabel="Total Revenue",
    ylabel="Countries",
    color="darkkhaki",
    alpha=0.9
)

top10_countries_by_quantity.plot.barh(
    ax=axis[1],
    title="Top 10 Countries by Quantity",
    xlabel="Total Quantity",
    ylabel="Countries",
    color="lightcoral",
    alpha=0.9
)

fig.subplots_adjust(hspace=0.3)

fig.savefig(IMAGES_DIR / "top10_countries.png", bbox_inches="tight")

![Top 10 Countries](../images/top10_countries.png)